In [ ]:
import kagglehub
import os
import pandas as pd
import matplotlib.pyplot as plt
from prophet import Prophet
from pykalman import KalmanFilter
from  xgboost import XGBRegressor
import optuna
import shap
from  sklearn.metrics import mean_absolute_error,mean_absolute_percentage_error

In [ ]:
path = kagglehub.dataset_download("robikscube/hourly-energy-consumption")
print("Path to dataset files:", path)
print('\n',os.listdir(path=path))

In [ ]:
files = os.listdir(path)
data_pool = {}

for filename in files:
    if filename.endswith('.csv'):
        full_path = os.path.join(path, filename)
        temp_df = pd.read_csv(full_path, header=0)
        
        if len(temp_df.columns) == 2:

            dataset_name = os.path.splitext(filename)[0]
            temp_df.columns = ['ds', 'y']
            
            kf = KalmanFilter(initial_state_mean=0, n_dim_obs=1)
            measurements = temp_df['y'].ffill().bfill().values
            state_means, _ = kf.filter(measurements)
        
            temp_df['y_kalman'] = state_means    
            data_pool[dataset_name] = temp_df
            
            print(f"{dataset_name} added to  dictionary")


In [ ]:
import matplotlib.pyplot as plt

for name, df in data_pool.items():

    plt.figure(figsize=(14,5))

    sample = df.iloc[:2000] 

    plt.plot(sample['ds'], sample['y'], label='Original', alpha=0.5)
    plt.plot(sample['ds'], sample['y_kalman'], label='Kalman', linewidth=2)

    plt.title(name)
    plt.legend()

    plt.show()

In [ ]:
all_forecasts = {}
performance_metrics = []

region_vars = data_pool.keys()

for var_name in region_vars:

    print(f"\n {var_name} processing...")

    df = data_pool[var_name].copy()
    df = df.iloc[24:].reset_index(drop=True)

    df['ds'] = pd.to_datetime(df['ds'])

    prophet_df = df[['ds', 'y_kalman']].rename(columns={'y_kalman': 'y'})

    model = Prophet(
        daily_seasonality=True,
        weekly_seasonality=True,
        yearly_seasonality=True
    )

    model.fit(prophet_df)

    future = model.make_future_dataframe(periods=24*7, freq='h')
    forecast = model.predict(future)

    all_forecasts[var_name] = forecast

    merged = prophet_df.merge(
        forecast[['ds', 'yhat']],
        on='ds',
        how='left'
    )

    merged['residual'] = merged['y'] - merged['yhat']

    split = int(len(merged) * 0.8)

    train = merged.iloc[:split].copy()
    test = merged.iloc[split:].copy()

    def create_features(df):
        df['hour'] = df['ds'].dt.hour
        df['dayofweek'] = df['ds'].dt.dayofweek
        df['month'] = df['ds'].dt.month

        df['lag_1'] = df['y'].shift(1)
        df['lag_24'] = df['y'].shift(24)
        df['rolling_mean_24'] = df['y'].rolling(24).mean()

        return df

    train = create_features(train)
    test = create_features(test)

    train = train.dropna()
    test = test.dropna()

    features = ['hour','dayofweek','month','lag_1','lag_24','rolling_mean_24']

    X_train = train[features]
    y_train = train['residual']

    X_test = test[features]
    y_test = test['residual']

    def objective(trial):

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 300, 1000),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "random_state": 42,
            "eval_metric": "mae",
            "tree_method": "hist"   
        }

        model = XGBRegressor(**params)

        model.fit(X_train, y_train)

        preds = model.predict(X_test)

        return mean_absolute_error(y_test, preds)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=15)

    best_params = study.best_params

    xgb = XGBRegressor(
                        **best_params,
                        random_state=42,
                        eval_metric='mae',
                        tree_method="hist",
                        device='cuda',
                        early_stopping_rounds=10
                    )

    xgb.fit(
                    X_train, y_train,
                    eval_set=[(X_test, y_test)],
                    verbose=False
                )

    print(f"Best iteration: {xgb.best_iteration}")

    residual_pred = xgb.predict(X_test)

    test['hybrid_pred'] = test['yhat'] + residual_pred

    mae_prophet = mean_absolute_error(test['y'], test['yhat'])
    mape_prophet = mean_absolute_percentage_error(test['y'], test['yhat'])

    mae_hybrid = mean_absolute_error(test['y'], test['hybrid_pred'])
    mape_hybrid = mean_absolute_percentage_error(test['y'], test['hybrid_pred'])

    performance_metrics.append({
                                    'dataset': var_name,
                                    'MAE_Prophet': mae_prophet,
                                    'MAPE_Prophet': mape_prophet,
                                    'MAE_Hybrid': mae_hybrid,
                                    'MAPE_Hybrid': mape_hybrid
                                })

    print(f"\n{var_name}")
    print(f"Prophet -> MAE: {mae_prophet:.4f}, MAPE: {mape_prophet:.4f}")
    print(f"Hybrid  -> MAE: {mae_hybrid:.4f}, MAPE: {mape_hybrid:.4f}")

    explainer = shap.TreeExplainer(xgb)
    shap_values = explainer.shap_values(X_test)

    print(f"\nSHAP Summary for {var_name}")
    shap.summary_plot(shap_values, X_test)